
# 01 — Data Exploration: PhysioNet EEG Motor Movement/Imagery

**Goal:** Understand the raw EEG data before preprocessing. We'll look at:
1. Dataset structure (subjects, runs, task paradigm)
2. Raw signal characteristics (sampling rate, channels, duration)
3. Time-domain visualization (what does raw EEG look like?)
4. Frequency-domain visualization (PSD — where is the power?)
5. Event/annotation structure (when do tasks happen?)
6. Obvious artifacts (eye blinks, line noise, drift)

**Dataset:** [EEG Motor Movement/Imagery Dataset](https://physionet.org/content/eegmmidb/1.0.0/)
- 109 subjects, 14 runs each
- 64-channel EEG @ 160 Hz
- Tasks: rest, real/imagined left/right fist, real/imagined both fists/feet

## Load Dataset

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from mne.datasets import eegbci
from mne.io import read_raw_edf

# Display settings
mne.set_log_level("WARNING")  
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 100

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print(f"MNE version: {mne.__version__}")

MNE version: 1.12.1


## Task paradigm

Each subject performed 14 runs:
- **Run 1:** Baseline, eyes open (1 min)
- **Run 2:** Baseline, eyes closed (1 min)
- **Runs 3, 7, 11:** Task 1 — open/close left or right fist (real movement)
- **Runs 4, 8, 12:** Task 2 — imagine opening/closing left or right fist
- **Runs 5, 9, 13:** Task 3 — open/close both fists or both feet (real)
- **Runs 6, 10, 14:** Task 4 — imagine opening/closing both fists or feet

**Event codes in annotations:**
- `T0` — rest
- `T1` — onset of motion (left fist OR both fists, depending on run)
- `T2` — onset of motion (right fist OR both feet, depending on run)

For our motor imagery classification, this will be focus on **runs 4, 8, 12** (imagined left vs right fist) — the classic BCI paradigm.

## Inspect Channel Layout & Montage

In [ ]:
# Download subject 1, runs 4, 8, 12 (imagined L/R fist)
subject = 1
runs = [4, 8, 12]

raw_fnames = eegbci.load_data(subject, runs, path=None, update_path=True)
print(f"Downloaded {len(raw_fnames)} files for subject {subject}:")
for f in raw_fnames:
    print(f"  {f}")

In [ ]:
# Start with run 4 only — get to know one file deeply before generalizing
raw = read_raw_edf(raw_fnames[0], preload=True, stim_channel="auto")

# Quick summary
print(raw)
print(f"\nSampling rate: {raw.info['sfreq']} Hz")
print(f"Duration: {raw.times[-1]:.1f} seconds ({raw.times[-1]/60:.2f} min)")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Number of samples: {raw.n_times}")

In [ ]:
print("Channel names (first 20):")
print(raw.ch_names[:20])
print(f"\nTotal: {len(raw.ch_names)} channels")

In [ ]:
# Strip trailing periods and standardize case
# eegbci provides a helper for exactly this
eegbci.standardize(raw)

# Set the standard 10-10 montage
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, on_missing="warn")

print("Cleaned channel names (first 20):")
print(raw.ch_names[:20])

## Visualize Raw Signals

*To be implemented in upcoming chunk.*

In [ ]:
raw.plot_sensors(show_names=True, sphere="auto")
plt.show()

## Event Structure

*To be implemented in upcoming chunk.*

In [ ]:
# Look at the annotations (task events)
print("Annotations:")
print(raw.annotations)
print(f"\nUnique event descriptions: {set(raw.annotations.description)}")
print(f"Total events: {len(raw.annotations)}")

# Convert to events array for inspection
events, event_id = mne.events_from_annotations(raw)
print(f"\nEvent ID mapping: {event_id}")
print(f"Events shape: {events.shape}  (n_events, [sample, prev_id, event_id])")
print(f"\nFirst 10 events:\n{events[:10]}")

In [ ]:
# Plot 10 seconds of raw data to see what we're dealing with
raw.plot(
    duration=10,
    n_channels=20,
    scalings={"eeg": 100e-6},  # 100 µV scale
    title=f"Subject {subject}, Run 4 — raw signal (first 10 s, 20 channels)",
    show=True,
)

In [ ]:
psd = raw.compute_psd(fmin=0.5, fmax=80, n_fft=2048)
fig = psd.plot(average=False, picks="eeg", show=False)
fig.suptitle(f"Subject {subject}, Run 4 — Power Spectral Density")
fig.tight_layout()
plt.show()

In [ ]:
psd = raw.compute_psd(fmin=1, fmax=45, n_fft=2048)
fig = psd.plot_topomap(bands=bands, normalize=True, show=False)
plt.show()

In [ ]:
# Plot frontal channels only — this is where blinks dominate
frontal_picks = ["Fp1", "Fp2", "Fpz", "Af7", "Af8"]
frontal_picks = [ch for ch in frontal_picks if ch in raw.ch_names]

raw.plot(
    duration=20,
    n_channels=len(frontal_picks),
    picks=frontal_picks,
    scalings={"eeg": 150e-6},
    title="Frontal channels — look for eye blinks (large slow deflections)",
    show=True,
)

## Summary

*To be implemented in upcoming chunk.*

In [ ]:
# Per-channel statistics: peak-to-peak, std, kurtosis
from scipy import stats

data = raw.get_data(picks="eeg") * 1e6  # convert to µV
ch_stats = pd.DataFrame({
    "channel": raw.ch_names,
    "mean_uV": data.mean(axis=1),
    "std_uV": data.std(axis=1),
    "ptp_uV": np.ptp(data, axis=1),
    "kurtosis": stats.kurtosis(data, axis=1),
})

# Sort by peak-to-peak — biggest values often = bad channels
ch_stats_sorted = ch_stats.sort_values("ptp_uV", ascending=False)
print("Top 10 channels by peak-to-peak amplitude (candidates for bad channels):")
print(ch_stats_sorted.head(10).to_string(index=False))

print("\nBottom 5 channels by peak-to-peak (candidates for flat/dead channels):")
print(ch_stats_sorted.tail(5).to_string(index=False))

In [ ]:
# Quick comparison: load all 3 runs and check basic stats are consistent
all_raws = []
for fname in raw_fnames:
    r = read_raw_edf(fname, preload=True, stim_channel="auto")
    eegbci.standardize(r)
    r.set_montage(montage, on_missing="warn")
    all_raws.append(r)

summary = pd.DataFrame([
    {
        "run": runs[i],
        "duration_s": r.times[-1],
        "n_channels": len(r.ch_names),
        "sfreq": r.info["sfreq"],
        "n_events": len(r.annotations),
    }
    for i, r in enumerate(all_raws)
])
print(summary.to_string(index=False))

## Takeaways for preprocessing

From this exploration:
1. **Channel names need cleaning** → use `eegbci.standardize()` and `standard_1005` montage
2. **Slow drift visible** → highpass filter at 1 Hz
3. **60 Hz line noise present** → notch filter at 60 Hz (and 120 Hz)
4. **Eye blinks on frontal channels** → ICA to remove
5. **Some channels have extreme peak-to-peak** → automatic bad-channel detection before ICA
6. **Sampling rate is 160 Hz** → Nyquist is 80 Hz, so our 1–40 Hz bandpass is well within bounds
7. **~30 events per run × 3 runs ≈ 90 trials per subject** for L/R imagined fist — adequate for subject-level classification

Next: notebook 02 implements this pipeline and saves cleaned epochs.